# 05 Model Serving Test

This notebook tests the registered champion models before or after deployment to Databricks Model Serving.

The project includes two registered models:

- Stockout Risk Classifier
- Reorder Flag Classifier

In [0]:
import mlflow
import pandas as pd

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
stockout_model_uri = "models:/workspace.retail_capstone.stockout_risk_classifier@champion"

stockout_model = mlflow.pyfunc.load_model(stockout_model_uri)

In [0]:
stockout_sample = pd.DataFrame([
    {
        "available_stock": 20.0,
        "avg_daily_sales": 8.5,
        "days_of_inventory_remaining": 2.35,
        "lead_time_days": 7.0,
        "reliability_score": 0.91,
        "category": "Home Decor",
        "warehouse_id": "WH001"
    },
    {
        "available_stock": 200.0,
        "avg_daily_sales": 3.0,
        "days_of_inventory_remaining": 66.67,
        "lead_time_days": 7.0,
        "reliability_score": 0.94,
        "category": "Home Decor",
        "warehouse_id": "WH001"
    }
])

stockout_predictions = stockout_model.predict(stockout_sample)

print(stockout_predictions)

['High Risk' 'Low Risk']


In [0]:
reorder_model_uri = "models:/workspace.retail_capstone.reorder_flag_classifier@champion"

reorder_model = mlflow.pyfunc.load_model(reorder_model_uri)

In [0]:
reorder_sample = pd.DataFrame([
    {
        "available_stock": 30.0,
        "reorder_level": 100.0,
        "reorder_quantity": 300.0,
        "lead_time_days": 7.0,
        "category": "Home Decor",
        "warehouse_id": "WH001"
    },
    {
        "available_stock": 250.0,
        "reorder_level": 100.0,
        "reorder_quantity": 300.0,
        "lead_time_days": 7.0,
        "category": "Home Decor",
        "warehouse_id": "WH001"
    }
])

reorder_predictions = reorder_model.predict(reorder_sample)

print(reorder_predictions)

['Reorder Needed' 'No Reorder Needed']


# Serving Test Summary

The champion models were loaded from Unity Catalog Model Registry using model aliases.

Model URIs:

- `models:/workspace.retail_capstone.stockout_risk_classifier@champion`
- `models:/workspace.retail_capstone.reorder_flag_classifier@champion`

Sample input data was passed to both models.

The stockout model predicts risk level.

The reorder model predicts whether reorder is needed.

This confirms that the registered champion models can be loaded and used for inference.

# Databricks Model Serving Endpoint Test

This section tests the deployed Databricks Model Serving endpoint for the stockout risk classifier.

Endpoint name:

`stockout-risk-classifier-endpoint`

In [0]:
import requests
import json
import os

In [0]:
DATABRICKS_HOST = "https://<your-databricks-workspace-url>"
DATABRICKS_TOKEN = "<your-databricks-personal-access-token>"

endpoint_name = "stockout-risk-classifier-endpoint"

url = f"{DATABRICKS_HOST}/serving-endpoints/{endpoint_name}/invocations"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "dataframe_records": [
        {
            "available_stock": 20,
            "avg_daily_sales": 8.5,
            "days_of_inventory_remaining": 2.35,
            "lead_time_days": 7,
            "reliability_score": 0.91,
            "category": "Home Decor",
            "warehouse_id": "WH001"
        },
        {
            "available_stock": 200,
            "avg_daily_sales": 3.0,
            "days_of_inventory_remaining": 66.67,
            "lead_time_days": 7,
            "reliability_score": 0.94,
            "category": "Home Decor",
            "warehouse_id": "WH001"
        }
    ]
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

print("Status code:", response.status_code)
print("Response:")
print(response.text)

Status code: 200
Response:
{"predictions": ["High Risk", "Low Risk"]}


# Example curl Request

The endpoint can also be tested using curl.

Replace `<your-databricks-workspace-url>` and `<your-token>` before running.

```bash
curl -X POST https://<your-databricks-workspace-url>/serving-endpoints/stockout-risk-classifier-endpoint/invocations \
  -H "Authorization: Bearer <your-token>" \
  -H "Content-Type: application/json" \
  -d '{
    "dataframe_records": [
      {
        "available_stock": 20,
        "avg_daily_sales": 8.5,
        "days_of_inventory_remaining": 2.35,
        "lead_time_days": 7,
        "reliability_score": 0.91,
        "category": "Home Decor",
        "warehouse_id": "WH001"
      }
    ]
  }'